# nn — entity-embedding dual-path

dual-path keras model. branch 1 = demographics + ses, branch 2 = healthcare access + behavioral. categoricals go through learned entity embeddings instead of one-hot (state_fips: 54 levels -> 8 dims). hp tuned with keras tuner hyperband.

_Adam_

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix, f1_score,
)
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    Dense, BatchNormalization, Activation, Dropout,
    Concatenate, Embedding, Flatten, Reshape,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau,
)
from tensorflow.keras.regularizers import l2

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Auto-detect environment
KAGGLE_DIR = Path("/kaggle/input/datasets/ajenks/brfss-2020-2024-cleaned-and-weighted")
LOCAL_DIR = Path("cleaned")
CLEAN_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else LOCAL_DIR
OUT_DIR = Path("/kaggle/working") if KAGGLE_DIR.exists() else LOCAL_DIR
ENV = "Kaggle" if KAGGLE_DIR.exists() else "Local"

print(f"Environment: {ENV}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

In [ ]:
df_train = pd.read_parquet(CLEAN_DIR / "brfss_train.parquet")
df_test = pd.read_parquet(CLEAN_DIR / "brfss_test.parquet")

print(f"Train: {df_train.shape[0]:,} rows x {df_train.shape[1]} cols")
print(f"Test:  {df_test.shape[0]:,} rows x {df_test.shape[1]} cols")
print(f"\nTarget balance (train): {df_train['PREV_HEALTH_HIGH'].mean():.3f} positive")
print(f"Target balance (test):  {df_test['PREV_HEALTH_HIGH'].mean():.3f} positive")

feature -> branch -> embedding dim follows `min(50, (n+1)//2)`. continuous + binary stay dense.

In [ ]:

CATEGORICAL_FEATURES = [
    # Branch 1: Demographic / Socioeconomic
    ("_AGE_G",       7,  3, 1),   # 1-6 → remap to 1-6, 0=missing
    ("_SEX",         3,  1, 1),   # 1-2 → remap to 1-2, 0=missing
    ("_IMPRACE",     7,  3, 1),   # 1-6
    ("MARITAL",      8,  3, 1),   # 1-7 (including possible 9=refused mapped to missing)
    ("_EDUCAG",      5,  2, 1),   # 1-4
    ("INCOME_GROUP", 6,  3, 1),   # 1-5
    ("EMPLOY1",     10,  4, 1),   # 1-8 + possible 9
    ("STATE_FIPS",  55,  8, 1),   # ~54 unique states/territories
    # Branch 2: Access / Behavioral
    ("SURVEY_YEAR",  6,  3, 2),   # 2020-2024 → remap to 1-5
]

# Branch 2 continuous/binary features (no embedding needed)
BRANCH2_DENSE_FEATURES = [
    "HAS_PERSONAL_DOCTOR",
    "HAS_HEALTH_PLAN",
    "HAS_HEALTH_PLAN_MISSING",
    "COST_BARRIER",
    "COST_BARRIER_MISSING",
    "HAS_DEPRESSION",
    "_BMI5_SCALED",
]

print(f"Categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"  Branch 1: {sum(1 for _,_,_,b in CATEGORICAL_FEATURES if b == 1)}")
print(f"  Branch 2: {sum(1 for _,_,_,b in CATEGORICAL_FEATURES if b == 2)}")
print(f"Branch 2 dense features: {len(BRANCH2_DENSE_FEATURES)}")
total_emb_params = sum(n * d for _, n, d, _ in CATEGORICAL_FEATURES)
print(f"Total embedding parameters: {total_emb_params:,}")

In [ ]:

def prepare_categorical_inputs(df_train, df_test, cat_features):
    """
    For each categorical feature, create integer-coded arrays suitable for
    Embedding layers. Missing values map to index 0.
    Returns dict of {col_name: (train_array, test_array)}.
    """
    cat_inputs = {}

    for col, n_cats, emb_dim, branch in cat_features:
        train_col = df_train[col].copy()
        test_col = df_test[col].copy()

        if col == "STATE_FIPS":
            # Remap FIPS codes to contiguous integers 1..N
            unique_fips = sorted(train_col.dropna().unique())
            fips_map = {fips: i + 1 for i, fips in enumerate(unique_fips)}
            train_col = train_col.map(fips_map)  # unmapped → NaN
            test_col = test_col.map(fips_map)

        elif col == "SURVEY_YEAR":
            # Remap 2020-2024 → 1-5
            year_map = {2020: 1, 2021: 2, 2022: 3, 2023: 4, 2024: 5}
            train_col = train_col.map(year_map)
            test_col = test_col.map(year_map)

        # Fill NaN with 0 (the "unknown" embedding index)
        train_arr = train_col.fillna(0).astype(np.int32).values
        test_arr = test_col.fillna(0).astype(np.int32).values

        # Clip to valid range [0, n_cats-1] to prevent index errors
        train_arr = np.clip(train_arr, 0, n_cats - 1)
        test_arr = np.clip(test_arr, 0, n_cats - 1)

        cat_inputs[col] = (train_arr, test_arr)
        print(f"  {col:20s}  unique(train): {len(np.unique(train_arr)):3d}  "
              f"range: [{train_arr.min()}, {train_arr.max()}]  "
              f"missing→0: {(train_arr == 0).sum():,}")

    return cat_inputs

print("Preparing categorical inputs:")
cat_inputs = prepare_categorical_inputs(df_train, df_test, CATEGORICAL_FEATURES)

In [ ]:

# Impute binary/categorical features with mode (fit on train only)
binary_cols = ["HAS_PERSONAL_DOCTOR", "HAS_HEALTH_PLAN", "COST_BARRIER", "HAS_DEPRESSION"]
imputer_mode = SimpleImputer(strategy="most_frequent")
df_train_b2 = df_train[BRANCH2_DENSE_FEATURES].copy()
df_test_b2 = df_test[BRANCH2_DENSE_FEATURES].copy()

df_train_b2[binary_cols] = imputer_mode.fit_transform(df_train_b2[binary_cols])
df_test_b2[binary_cols] = imputer_mode.transform(df_test_b2[binary_cols])

# Impute BMI with median + create missingness indicator
bmi_missing_train = df_train_b2["_BMI5_SCALED"].isna().astype(np.float32).values
bmi_missing_test = df_test_b2["_BMI5_SCALED"].isna().astype(np.float32).values

imputer_bmi = SimpleImputer(strategy="median")
df_train_b2[["_BMI5_SCALED"]] = imputer_bmi.fit_transform(df_train_b2[["_BMI5_SCALED"]])
df_test_b2[["_BMI5_SCALED"]] = imputer_bmi.transform(df_test_b2[["_BMI5_SCALED"]])

# Standardize BMI (the only truly continuous feature)
scaler_bmi = StandardScaler()
df_train_b2[["_BMI5_SCALED"]] = scaler_bmi.fit_transform(df_train_b2[["_BMI5_SCALED"]])
df_test_b2[["_BMI5_SCALED"]] = scaler_bmi.transform(df_test_b2[["_BMI5_SCALED"]])

# Add BMI missingness indicator
df_train_b2["_BMI5_MISSING"] = bmi_missing_train
df_test_b2["_BMI5_MISSING"] = bmi_missing_test

# Convert to float32 arrays
x_dense_train = df_train_b2.values.astype(np.float32)
x_dense_test = df_test_b2.values.astype(np.float32)
dense_col_names = list(df_train_b2.columns)

print(f"Dense Branch 2 features: {x_dense_train.shape[1]}")
print(f"  Columns: {dense_col_names}")
print(f"  NaN check — train: {np.isnan(x_dense_train).any()}, test: {np.isnan(x_dense_test).any()}")
print(f"  BMI missing (train): {bmi_missing_train.sum():,} ({bmi_missing_train.mean()*100:.1f}%)")

In [ ]:

def make_input_dict(cat_inputs, dense_arr, split="train"):
    """Build the {input_name: array} dict for model.fit() / model.predict()."""
    idx = 0 if split == "train" else 1
    d = {}
    for col, n_cats, emb_dim, branch in CATEGORICAL_FEATURES:
        d[f"cat_{col}"] = cat_inputs[col][idx]
    d["dense_branch2"] = dense_arr
    return d

X_train_dict = make_input_dict(cat_inputs, x_dense_train, "train")
X_test_dict = make_input_dict(cat_inputs, x_dense_test, "test")

# Targets and weights
y_train = df_train["PREV_HEALTH_HIGH"].values.astype(np.float32)
y_test = df_test["PREV_HEALTH_HIGH"].values.astype(np.float32)
w_train = df_train["sample_weight"].values.astype(np.float32)

print("Model input dict keys:")
for k, v in X_train_dict.items():
    print(f"  {k:25s}  shape: {v.shape}  dtype: {v.dtype}")

In [ ]:
def build_embedding_dual_path(
    cat_features=CATEGORICAL_FEATURES,
    n_dense_b2=8,  # number of dense Branch 2 features
    b1_width=128,
    b1_layers=2,
    b2_width=64,
    b2_layers=2,
    head_width=128,
    head_layers=2,
    dropout=0.3,
    l2_reg=1e-5,
    learning_rate=1e-3,
):
    """
    Build a dual-path binary classifier with entity embeddings for categoricals.

    Branch 1 receives all demographic/socioeconomic categoricals as embeddings.
    Branch 2 receives healthcare access + behavioral features (dense) plus
    SURVEY_YEAR as an embedding.

    Returns: compiled Keras Model
    """
    reg = l2(l2_reg) if l2_reg > 0 else None

    cat_input_layers = {}   # col_name → Input layer
    branch1_embeddings = []
    branch2_embeddings = []

    for col, n_cats, emb_dim, branch in cat_features:
        inp = Input(shape=(1,), name=f"cat_{col}", dtype="int32")
        cat_input_layers[col] = inp
        emb = Embedding(
            input_dim=n_cats,
            output_dim=emb_dim,
            name=f"emb_{col}",
        )(inp)
        flat = Flatten(name=f"flat_{col}")(emb)

        if branch == 1:
            branch1_embeddings.append(flat)
        else:
            branch2_embeddings.append(flat)

    b1 = Concatenate(name="b1_concat")(branch1_embeddings)
    for i in range(b1_layers):
        b1 = Dense(b1_width, kernel_regularizer=reg, name=f"b1_dense_{i}")(b1)
        b1 = BatchNormalization(name=f"b1_bn_{i}")(b1)
        b1 = Activation("relu", name=f"b1_relu_{i}")(b1)
        b1 = Dropout(dropout, name=f"b1_drop_{i}")(b1)

    dense_input = Input(shape=(n_dense_b2,), name="dense_branch2", dtype="float32")
    b2_parts = branch2_embeddings + [dense_input]
    b2 = Concatenate(name="b2_concat")(b2_parts)
    for i in range(b2_layers):
        b2 = Dense(b2_width, kernel_regularizer=reg, name=f"b2_dense_{i}")(b2)
        b2 = BatchNormalization(name=f"b2_bn_{i}")(b2)
        b2 = Activation("relu", name=f"b2_relu_{i}")(b2)
        b2 = Dropout(dropout, name=f"b2_drop_{i}")(b2)

    merged = Concatenate(name="merge")([b1, b2])
    h = merged
    width = head_width
    for i in range(head_layers):
        h = Dense(width, kernel_regularizer=reg, name=f"head_dense_{i}")(h)
        h = BatchNormalization(name=f"head_bn_{i}")(h)
        h = Activation("relu", name=f"head_relu_{i}")(h)
        h = Dropout(dropout, name=f"head_drop_{i}")(h)
        width = max(width // 2, 32)  # taper the head

    output = Dense(1, activation="sigmoid", name="preventive_health")(h)

    all_inputs = [cat_input_layers[col] for col, *_ in cat_features] + [dense_input]
    model = Model(inputs=all_inputs, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[AUC(name="auc"), "accuracy"],
    )

    return model

In [ ]:
model = build_embedding_dual_path(n_dense_b2=x_dense_train.shape[1])
model.summary()

stratified val split (10%); class weight folded into sample_weight; reduce-lr-on-plateau; early stopping patience 15.

In [ ]:
train_idx, val_idx = train_test_split(
    np.arange(len(y_train)),
    test_size=0.10,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

# Split each input array
X_fit = {k: v[train_idx] for k, v in X_train_dict.items()}
X_val = {k: v[val_idx] for k, v in X_train_dict.items()}
y_fit = y_train[train_idx]
y_val = y_train[val_idx]
w_fit = w_train[train_idx]
w_val = w_train[val_idx]

print(f"Fit:  {len(y_fit):,}  (positive rate: {y_fit.mean():.4f})")
print(f"Val:  {len(y_val):,}  (positive rate: {y_val.mean():.4f})")
print(f"Test: {len(y_test):,}  (positive rate: {y_test.mean():.4f})")

In [ ]:

from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0.0, 1.0])
cw = compute_class_weight("balanced", classes=classes, y=y_fit)
class_weight_map = {0: cw[0], 1: cw[1]}
print(f"Class weights: {class_weight_map}")

# Multiply: survey_weight * class_weight_for_that_row's_label
w_fit_adj = w_fit * np.where(y_fit == 1, class_weight_map[1], class_weight_map[0])
w_val_adj = w_val * np.where(y_val == 1, class_weight_map[1], class_weight_map[0])

print(f"Adjusted weight stats (fit):  mean={w_fit_adj.mean():.4f}  std={w_fit_adj.std():.4f}")
print(f"Adjusted weight stats (val):  mean={w_val_adj.mean():.4f}  std={w_val_adj.std():.4f}")

callbacks = [
    EarlyStopping(
        monitor="val_auc", mode="max", patience=15,
        restore_best_weights=True, verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_auc", mode="max", factor=0.5,
        patience=5, min_lr=1e-6, verbose=1,
    ),
    ModelCheckpoint(
        filepath=str(OUT_DIR / "nn_embedding_best.keras"),
        monitor="val_auc", mode="max",
        save_best_only=True, verbose=1,
    ),
]

In [ ]:
history = model.fit(
    X_fit, y_fit,
    sample_weight=w_fit_adj,
    validation_data=(X_val, y_val, w_val_adj),
    epochs=60,
    batch_size=1024,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
y_prob = model.predict(X_test_dict, batch_size=2048, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# Metrics
auc_roc = roc_auc_score(y_test, y_prob)
auc_pr = average_precision_score(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)
f1 = f1_score(y_test, y_pred)

print("=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
print(f"  AUC-ROC:          {auc_roc:.4f}")
print(f"  AUC-PR:           {auc_pr:.4f}")
print(f"  Brier Score:      {brier:.4f}")
print(f"  F1 Score:         {f1:.4f}")
print(f"  Accuracy:         {(y_pred == y_test).mean():.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Low (0-2)", "High (3-4)"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# AUC
axes[0].plot(history.history["auc"], label="Train AUC")
axes[0].plot(history.history["val_auc"], label="Val AUC")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("AUC")
axes[0].set_title("AUC-ROC")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history["loss"], label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Val Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Binary Cross-Entropy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
if "lr" in history.history:
    axes[2].plot(history.history["lr"])
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Learning Rate")
    axes[2].set_title("LR Schedule")
    axes[2].set_yscale("log")
    axes[2].grid(True, alpha=0.3)
else:
    axes[2].text(0.5, 0.5, "LR history\nnot available", ha="center", va="center")

plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_training_history.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name="NN (Embeddings)")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_title(f"ROC Curve (AUC = {auc_roc:.4f})")
axes[0].grid(True, alpha=0.3)

PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name="NN (Embeddings)")
axes[1].axhline(y=y_test.mean(), color="k", linestyle="--", alpha=0.3, label="Baseline")
axes[1].set_title(f"Precision-Recall Curve (AP = {auc_pr:.4f})")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_roc_pr_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

hp search runs ~30-60 min on a kaggle gpu. set MAX_TRIALS lower to smoke-test.

In [ ]:
import keras_tuner as kt

def build_tunable_model(hp):
    """Keras Tuner model builder — searches over architecture hyperparameters."""
    return build_embedding_dual_path(
        n_dense_b2=x_dense_train.shape[1],
        b1_width=hp.Choice("b1_width", [64, 128, 256]),
        b1_layers=hp.Int("b1_layers", 1, 3),
        b2_width=hp.Choice("b2_width", [32, 64, 128]),
        b2_layers=hp.Int("b2_layers", 1, 2),
        head_width=hp.Choice("head_width", [64, 128, 256]),
        head_layers=hp.Int("head_layers", 1, 3),
        dropout=hp.Float("dropout", 0.15, 0.5, step=0.05),
        l2_reg=hp.Choice("l2_reg", [0.0, 1e-5, 1e-4]),
        learning_rate=hp.Choice("lr", [3e-4, 1e-3, 3e-3]),
    )

tuner = kt.Hyperband(
    build_tunable_model,
    objective=kt.Objective("val_auc", direction="max"),
    max_epochs=30,
    factor=3,
    hyperband_iterations=2,
    directory=str(OUT_DIR / "kt_logs"),
    project_name="nn_embedding_search",
    overwrite=True,
    seed=RANDOM_STATE,
)

tuner.search_space_summary()

In [ ]:
tuner_callbacks = [
    EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=3, min_lr=1e-6),
]

tuner.search(
    X_fit, y_fit,
    sample_weight=w_fit_adj,
    validation_data=(X_val, y_val, w_val_adj),
    batch_size=1024,
    callbacks=tuner_callbacks,
    verbose=2,
)

print("\nBest hyperparameters:")
best_hp = tuner.get_best_hyperparameters(1)[0]
for param in ["b1_width", "b1_layers", "b2_width", "b2_layers",
              "head_width", "head_layers", "dropout", "l2_reg", "lr"]:
    print(f"  {param}: {best_hp.get(param)}")

branch ablation = nonlinear analog of sq2's lr test. train branch1-only, branch2-only, full; compare aucs.

In [ ]:
eval_model = best_model if "best_model" in dir() else model

# Branch 1 categorical features vs Branch 2
b1_cat_cols = [col for col, _, _, b in CATEGORICAL_FEATURES if b == 1]
b2_cat_cols = [col for col, _, _, b in CATEGORICAL_FEATURES if b == 2]

def ablated_predict(model, X_dict, zero_branch):
    """
    Predict with one branch zeroed out.
    zero_branch=1 → zero all Branch 1 (demo/socio) inputs
    zero_branch=2 → zero all Branch 2 (access/behavioral) inputs
    """
    X_ablated = {}
    for k, v in X_dict.items():
        if zero_branch == 1:
            # Zero out Branch 1 categoricals
            if any(k == f"cat_{col}" for col in b1_cat_cols):
                X_ablated[k] = np.zeros_like(v)  # maps to the "missing" embedding
            else:
                X_ablated[k] = v
        elif zero_branch == 2:
            # Zero out Branch 2 categoricals + dense
            if any(k == f"cat_{col}" for col in b2_cat_cols):
                X_ablated[k] = np.zeros_like(v)
            elif k == "dense_branch2":
                X_ablated[k] = np.zeros_like(v)
            else:
                X_ablated[k] = v
    return model.predict(X_ablated, batch_size=2048, verbose=0).ravel()

# Full model predictions
y_prob_full = eval_model.predict(X_test_dict, batch_size=2048, verbose=0).ravel()

# Branch 2 only (zero out Branch 1)
y_prob_b2_only = ablated_predict(eval_model, X_test_dict, zero_branch=1)

# Branch 1 only (zero out Branch 2)
y_prob_b1_only = ablated_predict(eval_model, X_test_dict, zero_branch=2)

auc_full = roc_auc_score(y_test, y_prob_full)
auc_b1_only = roc_auc_score(y_test, y_prob_b1_only)
auc_b2_only = roc_auc_score(y_test, y_prob_b2_only)

print("Branch Ablation Results (AUC-ROC on test set):")
print(f"  Full model (both branches):    {auc_full:.4f}")
print(f"  Branch 1 only (Demo/Socio):    {auc_b1_only:.4f}  (drop: {auc_full - auc_b1_only:+.4f})")
print(f"  Branch 2 only (Access/Behav):  {auc_b2_only:.4f}  (drop: {auc_full - auc_b2_only:+.4f})")
print(f"\n  Branch 1 contribution: {auc_full - auc_b2_only:.4f} AUC points")
print(f"  Branch 2 contribution: {auc_full - auc_b1_only:.4f} AUC points")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = ["Full Model\n(Both Branches)", "Branch 1 Only\n(Demo/Socio)", "Branch 2 Only\n(Access/Behav)"]
aucs = [auc_full, auc_b1_only, auc_b2_only]
colors = ["#2ecc71", "#3498db", "#e74c3c"]
bars = ax.bar(labels, aucs, color=colors, edgecolor="black", linewidth=0.5)

for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f"{val:.4f}", ha="center", va="bottom", fontweight="bold")

ax.set_ylabel("AUC-ROC")
ax.set_title("Branch Ablation: Contribution of Each Input Branch")
ax.set_ylim(0.5, max(aucs) + 0.03)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.3, label="Random")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_branch_ablation.png"), dpi=150, bbox_inches="tight")
plt.show()

shap.DeepExplainer on a 2k-row test subsample (full set is too slow).

In [ ]:
import shap

N_SHAP = 500
N_BG = 100
np.random.seed(RANDOM_STATE)
shap_idx = np.random.choice(len(y_test), size=N_SHAP, replace=False)
bg_idx = np.random.choice(len(y_train), size=N_BG, replace=False)

input_keys = list(X_test_dict.keys())

def stack_inputs(X_dict, idx):
    """Horizontally stack all input arrays into one float32 matrix."""
    parts = []
    for k in input_keys:
        arr = X_dict[k][idx]
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)
        parts.append(arr.astype(np.float32))
    return np.hstack(parts)

X_shap_flat = stack_inputs(X_test_dict, shap_idx)
X_bg_flat = stack_inputs(X_train_dict, bg_idx)

# Column boundaries for splitting back into the dict
col_splits = []
for k in input_keys:
    sample = X_test_dict[k][:1]
    col_splits.append(1 if sample.ndim == 1 else sample.shape[1] if sample.ndim == 2 else 1)

def predict_from_flat(X_flat):
    """Split a flat array back into a LIST of arrays (matching model.inputs order)."""
    inputs_list = []
    col = 0
    for k, width in zip(input_keys, col_splits):
        chunk = X_flat[:, col:col + width]
        if "cat_" in k:
            inputs_list.append(chunk.astype(np.int32))
        else:
            inputs_list.append(chunk.astype(np.float32))
        col += width
    return eval_model.predict(inputs_list, batch_size=2048, verbose=0).ravel()

# Sanity check: wrapper matches direct prediction
direct_inputs = [X_test_dict[k][shap_idx[:5]].reshape(-1, 1)
                 if X_test_dict[k][shap_idx[:5]].ndim == 1
                 else X_test_dict[k][shap_idx[:5]]
                 for k in input_keys]
direct = eval_model.predict(direct_inputs, batch_size=5, verbose=0).ravel()
wrapped = predict_from_flat(X_shap_flat[:5])
assert np.allclose(direct, wrapped, atol=1e-5), "Wrapper mismatch!"
print("Sanity check passed.")

# Feature names (one per column in the flat matrix)
feature_names = []
for k, width in zip(input_keys, col_splits):
    if k == "dense_branch2":
        feature_names.extend(dense_col_names)
    else:
        feature_names.append(k.replace("cat_", ""))

print(f"SHAP sample: {N_SHAP} test rows, {N_BG} background rows")
print(f"Flat input shape: {X_shap_flat.shape}")
print(f"Features ({len(feature_names)}): {feature_names}")

In [ ]:

explainer = shap.KernelExplainer(predict_from_flat, X_bg_flat)
shap_values_raw = explainer.shap_values(X_shap_flat, nsamples=200)

shap_matrix = shap_values_raw
print(f"SHAP matrix shape: {shap_matrix.shape}")
print(f"Feature names ({len(feature_names)}): {feature_names}")

In [ ]:
mean_abs_shap = np.abs(shap_matrix).mean(axis=0)
sorted_idx = np.argsort(mean_abs_shap)[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(feature_names))
ax.barh(y_pos, mean_abs_shap[sorted_idx[::-1]],
        color="#3498db", edgecolor="black", linewidth=0.3)
ax.set_yticks(y_pos)
ax.set_yticklabels([feature_names[i] for i in sorted_idx[::-1]])
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Feature Importance — Neural Network (Entity Embeddings)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_shap_importance.png"), dpi=150, bbox_inches="tight")
plt.show()

# Print ranked importance
print("\nFeature importance ranking (mean |SHAP|):")
for rank, idx in enumerate(sorted_idx, 1):
    print(f"  {rank:2d}. {feature_names[idx]:30s}  {mean_abs_shap[idx]:.5f}")

pull the learned embeddings and check what the model thinks is similar.

In [ ]:
from sklearn.decomposition import PCA

def get_embedding_weights(model, layer_name):
    """Extract the learned embedding matrix from a named layer."""
    return model.get_layer(layer_name).get_weights()[0]

# STATE_FIPS embedding (the most interesting one — 54 states → 8 dims)
state_emb = get_embedding_weights(eval_model, "emb_STATE_FIPS")
print(f"STATE_FIPS embedding shape: {state_emb.shape}")

# PCA to 2D for visualization
pca = PCA(n_components=2)
state_2d = pca.fit_transform(state_emb[1:])  # skip index 0 (missing bucket)

# FIPS code lookup for labels
unique_fips = sorted(df_train["STATE_FIPS"].dropna().unique())
fips_to_abbr = {
    1:"AL",2:"AK",4:"AZ",5:"AR",6:"CA",8:"CO",9:"CT",10:"DE",11:"DC",12:"FL",
    13:"GA",15:"HI",16:"ID",17:"IL",18:"IN",19:"IA",20:"KS",21:"KY",22:"LA",
    23:"ME",24:"MD",25:"MA",26:"MI",27:"MN",28:"MS",29:"MO",30:"MT",31:"NE",
    32:"NV",33:"NH",34:"NJ",35:"NM",36:"NY",37:"NC",38:"ND",39:"OH",40:"OK",
    41:"OR",42:"PA",44:"RI",45:"SC",46:"SD",47:"TN",48:"TX",49:"UT",50:"VT",
    51:"VA",53:"WA",54:"WV",55:"WI",56:"WY",66:"GU",72:"PR",78:"VI",
}

fig, ax = plt.subplots(figsize=(12, 8))
for i, fips in enumerate(unique_fips):
    if i < len(state_2d):
        abbr = fips_to_abbr.get(int(fips), str(int(fips)))
        ax.scatter(state_2d[i, 0], state_2d[i, 1], s=30, zorder=5)
        ax.annotate(abbr, (state_2d[i, 0], state_2d[i, 1]),
                    fontsize=7, ha="center", va="bottom")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("Learned STATE_FIPS Embeddings (PCA projection)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_state_embeddings.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

emb_configs = [
    ("emb__AGE_G", "_AGE_G", {1:"18-24",2:"25-34",3:"35-44",4:"45-54",5:"55-64",6:"65+"}),
    ("emb_EMPLOY1", "EMPLOY1", {1:"Employed",2:"Self-emp",3:"Unemp>1y",4:"Unemp<1y",
                                 5:"Homemaker",6:"Student",7:"Retired",8:"Unable"}),
    ("emb__IMPRACE", "_IMPRACE", {1:"White",2:"Black",3:"Asian",4:"AI/AN",5:"Hispanic",6:"Other"}),
]

for ax, (layer_name, col, label_map) in zip(axes, emb_configs):
    emb_w = get_embedding_weights(eval_model, layer_name)
    # Skip index 0 (missing bucket), plot indices 1..N
    n_cats = len(label_map)
    emb_sub = emb_w[1:n_cats+1]

    if emb_sub.shape[1] >= 2:
        ax.scatter(emb_sub[:, 0], emb_sub[:, 1], s=60, zorder=5)
        for i, (code, label) in enumerate(sorted(label_map.items())):
            if i < len(emb_sub):
                ax.annotate(label, (emb_sub[i, 0], emb_sub[i, 1]),
                            fontsize=7, ha="center", va="bottom")
        ax.set_xlabel("Dim 1")
        ax.set_ylabel("Dim 2")
    else:
        # 1D embedding — plot as number line
        vals = emb_sub.ravel()
        ax.scatter(vals, np.zeros_like(vals), s=60, zorder=5)
        for i, (code, label) in enumerate(sorted(label_map.items())):
            if i < len(vals):
                ax.annotate(label, (vals[i], 0), fontsize=7,
                            ha="center", va="bottom", rotation=45)
        ax.set_xlabel("Embedding value")
        ax.set_yticks([])

    ax.set_title(f"{col} Embedding")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_embeddings_other.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
nn_preds = pd.DataFrame({
    "y_true": y_test,
    "y_prob_nn": y_prob_full,
    "y_pred_nn": (y_prob_full >= 0.5).astype(int),
}, index=df_test.index)
nn_preds.to_parquet(OUT_DIR / "nn_test_predictions.parquet")

shap_df = pd.DataFrame(shap_matrix, columns=feature_names)
shap_df.to_parquet(OUT_DIR / "nn_shap_values.parquet")

eval_model.save(str(OUT_DIR / "nn_embedding_final.keras"))

print("Saved outputs:")
print(f"  {OUT_DIR}/nn_test_predictions.parquet  ({len(nn_preds):,} rows)")
print(f"  {OUT_DIR}/nn_shap_values.parquet       ({shap_df.shape})")
print(f"  {OUT_DIR}/nn_embedding_final.keras")
print(f"  {OUT_DIR}/nn_training_history.png")
print(f"  {OUT_DIR}/nn_roc_pr_curves.png")
print(f"  {OUT_DIR}/nn_branch_ablation.png")
print(f"  {OUT_DIR}/nn_shap_importance.png")
print(f"  {OUT_DIR}/nn_state_embeddings.png")
print(f"  {OUT_DIR}/nn_embeddings_other.png")

print(f"\n{'='*50}")
print(f"FINAL METRICS SUMMARY")
print(f"{'='*50}")
print(f"  Default model AUC-ROC: {auc_roc:.4f}")
if "auc_roc_tuned" in dir():
    print(f"  Tuned model AUC-ROC:   {auc_roc_tuned:.4f}")
print(f"  Baseline notebook AUC: ~0.736 (val, from training logs)")